In [1]:
import pandas as pd
import logging
from joblib import Parallel, delayed




from rdkit import Chem
from smiles_blocks.files import INTERIM_DATA_DIR,EXTERNAL_DATA_DIR, PROCESSED_DATA_DIR, FIGURES_DIR

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)



In [2]:
# Define paths
block_library_dir = INTERIM_DATA_DIR / "well_sampled" / "block_library"
df_list = [pd.read_parquet(file) for file in block_library_dir.glob("*.parquet")]
df  = pd.concat(df_list, ignore_index=True)
block2unique_id = {block: unique_id for unique_id, block in zip(df["unique_id"], df["block"])}

In [3]:
moses_df = pd.read_csv(EXTERNAL_DATA_DIR / "moses_dataset.csv" )

In [4]:
def smi2cansmiles(smi: str) -> str | None:
    mol = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(mol, canonical=True) if mol else None

canonical = Parallel(n_jobs=-1, backend="loky", batch_size=500)(  # pyright: ignore[reportArgumentType]
    delayed(smi2cansmiles)(smi) for smi in moses_df['SMILES']
)

In [5]:
moses_df['canonical_smiles'] = canonical

In [6]:
def smi2scaffolds(smi: str ,kekulize: bool = True,allBondsExplicit: bool = True) -> str | None:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        scaffold = Chem.MurckoDecompose(mol)
        return Chem.MolToSmiles(scaffold, canonical=True, kekuleSmiles=kekulize,allBondsExplicit=allBondsExplicit)
    return None

scaffolds = Parallel(n_jobs=-1, backend="loky", batch_size=500)(  # pyright: ignore[reportArgumentType]
    delayed(smi2scaffolds)(smi, kekulize=False, allBondsExplicit=False) for smi in moses_df['canonical_smiles']
)
moses_df['scaffold'] = scaffolds

In [7]:
moses_df

,SMILES,SPLIT,canonical_smiles,scaffold
0,CCCS(=O)c1ccc2[nH]c(=NC(=O)OC)[nH]c2c1,train,CCCS(=O)c1ccc2[nH]c(=NC(=O)OC)[nH]c2c1,N=c1[nH]c2ccccc2[nH]1
1,CC(C)(C)C(=O)C(Oc1ccc(Cl)cc1)n1ccnc1,train,CC(C)(C)C(=O)C(Oc1ccc(Cl)cc1)n1ccnc1,c1ccc(OCn2ccnc2)cc1
2,CC1C2CCC(C2)C1CN(CCO)C(=O)c1ccc(Cl)cc1,test,CC1C2CCC(C2)C1CN(CCO)C(=O)c1ccc(Cl)cc1,O=C(NCC1CC2CCC1C2)c1ccccc1
3,Cc1c(Cl)cccc1Nc1ncccc1C(=O)OCC(O)CO,train,Cc1c(Cl)cccc1Nc1ncccc1C(=O)OCC(O)CO,c1ccc(Nc2ccccn2)cc1
4,Cn1cnc2c1c(=O)n(CC(O)CO)c(=O)n2C,train,Cn1cnc2c1c(=O)n(CC(O)CO)c(=O)n2C,O=c1[nH]c(=O)c2[nH]cnc2[nH]1
...,...,...,...,...
1936957,N#Cc1c(Br)cnc(N)c1Br,train,N#Cc1c(Br)cnc(N)c1Br,c1ccncc1
1936958,COC(=O)c1cc(CNC(=O)OC(C)(C)C)ccc1C,train,COC(=O)c1cc(CNC(=O)OC(C)(C)C)ccc1C,c1ccccc1
1936959,NC(=O)c1ccc2ccccc2c1Br,train,NC(=O)c1ccc2ccccc2c1Br,c1ccc2ccccc2c1
1936960,CC(=O)Nc1cccc(-c2nc3cc(C)ccc3[nH]c2=O)c1,train,CC(=O)Nc1cccc(-c2nc3cc(C)ccc3[nH]c2=O)c1,O=c1[nH]c2ccccc2nc1-c1ccccc1


In [8]:
blocked_smiles_dir = INTERIM_DATA_DIR / "well_sampled" / "blocked_smiles"
df_list = [pd.read_parquet(file) for file in blocked_smiles_dir.glob("*.parquet")]
df_blocked_smiles = pd.concat(df_list, ignore_index=True)

In [9]:
successful_blocking = df_blocked_smiles[df_blocked_smiles["fragmentation_success"]]

In [10]:
successful_blocking["canonical_smiles"] = Parallel(n_jobs=-1, backend="loky", batch_size=500)(  # pyright: ignore[reportArgumentType]
    delayed(smi2cansmiles)(smi) for smi in successful_blocking['smiles']
)

In [11]:
successful_blocking["scaffold"] = Parallel(n_jobs=-1, backend="loky", batch_size=500)(  # pyright: ignore[reportArgumentType]
    delayed(smi2scaffolds)(smi, kekulize=True, allBondsExplicit=True) for smi in successful_blocking['canonical_smiles']
)

In [12]:
successful_blocking

,ref_smiles,retrosynthetic_success,fragmentation_success,smiles,smiles_blocked,mem_score,unique_id_seq,retro_bond_ratio,nb_block_cq_ok,canonical_smiles,scaffold
0,Cc1ccc(C2CC(c3ccccc3)n3ncnc3N2)cc1,True,True,C1(-N-C2-N(-N=C-N=2)-C(-C-1)-C1=C-C=C-C=C-1)-C...,C1(-N-C2-N(-N=C-N=2)-C(-C-1)-C1=C-C=C-C=C-1).C...,1.952381,6_c1ccc(C2CCNc3ncnn32)cc1_6.4_Cc1ccccc1_0,1.0,1,Cc1ccc(C2CC(c3ccccc3)n3ncnc3N2)cc1,C1=C-C=C(-C2-C-C(-C3=C-C=C-C=C-3)-N3-N=C-N=C-3...
1,O=C1CC(C(=O)Nc2cccc(F)c2)n2ncnc2N1,True,True,C1(-N2-N=C-N=C-2-N-C(=O)-C-1)-C(=O)-N-C1-C=C(-...,C1(-N2-N=C-N=C-2-N-C(=O)-C-1)-C(=O).N.C1-C=C(-...,1.607143,2_O=CC1CC(=O)Nc2ncnn21_1.0_N_0.3_Fc1ccccc1_0,1.0,2,O=C1CC(C(=O)Nc2cccc(F)c2)n2ncnc2N1,O=C1-C-C(-C(=O)-N-C2=C-C=C-C=C-2)-N2-N=C-N=C-2...
2,CCCC1=C(C(=O)OCC)C(c2ccc(F)cc2)n2ncnc2N1,True,True,C(-C)-O-C(=O)-C1=C(-N-C2-N(-N=C-N=2)-C-1-C1-C=...,C(-C).O.C(=O).C1=C(-N-C2-N(-N=C-N=2)-C-1-C1-C=...,1.681818,0_CC_0.0_O_0.0_C=O_0.6_Fc1ccc(C2C=CNc3ncnn32)c...,1.0,4,CCCC1=C(C(=O)OCC)C(c2ccc(F)cc2)n2ncnc2N1,C1=C-C(-C2=C-C=C-C=C-2)-N2-N=C-N=C-2-N-1
3,O=C(Nc1ccc(Br)cc1)c1c[nH]cn1,True,True,N1=C-N-C=C-1-C(=O)-N-C1=C-C=C(-C=C-1)-Br,N1=C-N-C=C-1.C(=O).N.C1=C-C=C(-C=C-1)-Br,0.846154,4_c1c[nH]cn1_0.0_C=O_0.0_N_0.4_Brc1ccccc1_0,1.0,4,O=C(Nc1ccc(Br)cc1)c1c[nH]cn1,O=C(-N-C1=C-C=C-C=C-1)-C1=C-N-C=N-1
4,CC1=C(C(N)=O)C(c2ccccc2)n2nc(-c3cccs3)nc2N1,True,True,C1=C-S-C(=C-1)-C1=N-N2-C(-C3=C-C=C-C=C-3)-C(=C...,C1=C-S-C(=C-1).C1=N-N2-C(-C3=C-C=C-C=C-3)-C(=C...,2.086957,0_c1ccsc1_2.12_CC1=CC(c2ccccc2)n2ncnc2N1_2.1_N...,1.0,2,CC1=C(C(N)=O)C(c2ccccc2)n2nc(-c3cccs3)nc2N1,C1=C-C(-C2=C-C=C-C=C-2)-N2-N=C(-C3=C-C=C-S-3)-...
...,...,...,...,...,...,...,...,...,...,...,...
1936957,Cc1cc(NC(=O)COc2ccccc2)nn1Cc1ccccc1,True,True,C1=C-C(=C-C=C-1)-O-C-C(=O)-N-C1=N-N(-C(-C)=C-1...,C1=C-C(=C-C=C-1).O.C-C(=O).N.C1=N-N(-C(-C)=C-1...,1.031250,0_c1ccccc1_2.0_O_0.0_CC=O_1.0_N_0.3_Cc1ccn[nH]...,1.0,7,Cc1cc(NC(=O)COc2ccccc2)nn1Cc1ccccc1,O=C(-C-O-C1=C-C=C-C=C-1)-N-C1=N-N(-C-C2=C-C=C-...
1936958,O=C(Nc1cc(Cl)ccc1Cl)c1ccc(-n2cnnn2)cc1,True,True,Cl-C1=C-C=C(-Cl)-C(=C-1)-N-C(=O)-C1-C=C-C(=C-C...,Cl-C1=C-C=C(-Cl)-C(=C-1).N.C(=O).C1-C=C-C(=C-C...,0.966667,0_Clc1ccc(Cl)cc1_6.0_N_0.0_C=O_0.0_c1ccccc1_3....,1.0,4,O=C(Nc1cc(Cl)ccc1Cl)c1ccc(-n2cnnn2)cc1,O=C(-N-C1=C-C=C-C=C-1)-C1=C-C=C(-N2-C=N-N=N-2)...
1936959,O=C(Nc1ccccc1F)c1ccc(-n2cnnn2)cc1,True,True,C1=C(-F)-C(=C-C=C-1)-N-C(=O)-C1-C=C-C(=C-C=1)-...,C1=C(-F)-C(=C-C=C-1).N.C(=O).C1-C=C-C(=C-C=1)....,1.068966,6_Fc1ccccc1_2.0_N_0.0_C=O_0.0_c1ccccc1_3.4_c1n...,1.0,4,O=C(Nc1ccccc1F)c1ccc(-n2cnnn2)cc1,O=C(-N-C1=C-C=C-C=C-1)-C1=C-C=C(-N2-C=N-N=N-2)...
1936960,CCc1ccc(NC(=O)c2ccc(-n3cnnn3)cc2)cc1,True,True,C1-N(-N=N-N=1)-C1=C-C=C(-C=C-1)-C(=O)-N-C1-C=C...,C1-N(-N=N-N=1).C1=C-C=C(-C=C-1).C(=O).N.C1-C=C...,1.050000,0_c1nnn[nH]1_4.0_c1ccccc1_3.0_C=O_0.0_N_0.0_c1...,1.0,5,CCc1ccc(NC(=O)c2ccc(-n3cnnn3)cc2)cc1,O=C(-N-C1=C-C=C-C=C-1)-C1=C-C=C(-N2-C=N-N=N-2)...


## Data Preparation for MolGPT Training

In [13]:
# Add train/valid/test split from original moses dataset
split_map = moses_df.set_index("SMILES")["SPLIT"].str.lower().to_dict()
successful_blocking = successful_blocking.copy()
successful_blocking["split"] = successful_blocking["ref_smiles"].map(split_map)
print(successful_blocking["split"].value_counts())
print(f"Missing splits: {successful_blocking['split'].isna().sum():,}")

split
train             1577803
test_scaffolds     175731
test               175321
Name: count, dtype: int64
Missing splits: 0


In [14]:
# Deduplicate block library on unique_id to get one canonical block per unique_id
# Multiple blocks can share the same unique_id (same molecule + same anchor points)
# keeping the first one as canonical collapses them into a single token
deduplicate_df = df.drop_duplicates(subset="unique_id", keep="first").reset_index(drop=True)
print(f"Full block library: {len(df):,} rows")
print(f"After dedup on unique_id: {len(deduplicate_df):,} rows")

unique_id_to_block = dict(zip(deduplicate_df["unique_id"], deduplicate_df["block"]))
print(f"unique_id_to_block size: {len(unique_id_to_block):,}")
print(f"block2unique_id size:    {len(block2unique_id):,}")

Full block library: 10,612,101 rows
After dedup on unique_id: 119,645 rows
unique_id_to_block size: 119,645
block2unique_id size:    266,621


In [15]:
def normalize_smiles_blocked(smi: str) -> str | None:
    """Map each block -> unique_id -> canonical block to collapse equivalent representations."""
    try:
        return ".".join(unique_id_to_block[block2unique_id[b]] for b in smi.split("."))
    except KeyError:
        return None

successful_blocking["smiles_blocked_normalized"] = successful_blocking["smiles_blocked"].map(
    normalize_smiles_blocked
)
n_failed = successful_blocking["smiles_blocked_normalized"].isna().sum()
print(f"Failed normalizations: {n_failed:,} / {len(successful_blocking):,}")

Failed normalizations: 0 / 1,928,855


## Classic Mode CSV

In [16]:
molgpt_dir = PROCESSED_DATA_DIR / "molgpt_data"
molgpt_dir.mkdir(parents=True, exist_ok=True)

classic_df = (
    successful_blocking[["canonical_smiles", "scaffold", "split"]]
    .rename(columns={"canonical_smiles": "smiles", "scaffold": "scaffold_smiles"})
    .dropna(subset=["smiles", "split"])
)
output_path = molgpt_dir / "molgpt_classic.csv"
classic_df.to_csv(output_path, index=False)
print(f"Written {len(classic_df):,} rows to {output_path}")
classic_df["split"].value_counts()

Written 1,928,855 rows to /Users/ereboul/projects/smiles_blocks/data/processed/molgpt_data/molgpt_classic.csv


split
train             1577803
test_scaffolds     175731
test               175321
Name: count, dtype: int64

## Block Mode CSV

In [17]:
block_df = (
    successful_blocking[["smiles_blocked_normalized", "split"]]
    .rename(columns={"smiles_blocked_normalized": "smiles"})
    .dropna(subset=["smiles", "split"])
)
output_path = molgpt_dir / "molgpt_block.csv"
block_df.to_csv(output_path, index=False)
print(f"Written {len(block_df):,} rows to {output_path}")
print(block_df["split"].value_counts())

# Collect unique block tokens that actually appear in the dataset
all_blocks_in_data = set()
for smi in block_df["smiles"]:
    all_blocks_in_data.update(smi.split("."))
print(f"\nUnique block tokens in dataset: {len(all_blocks_in_data):,}")
print(f"Deduplicated unique_id vocab:   {len(unique_id_to_block):,}")

# Write vocab file (one token per line, sorted for reproducibility)
vocab_path = molgpt_dir / "block_vocab.txt"
with vocab_path.open("w") as f:
    f.write("\n".join(sorted(all_blocks_in_data)))
print(f"Vocab file written to {vocab_path}")

Written 1,928,855 rows to /Users/ereboul/projects/smiles_blocks/data/processed/molgpt_data/molgpt_block.csv
split
train             1577803
test_scaffolds     175731
test               175321
Name: count, dtype: int64

Unique block tokens in dataset: 119,645
Deduplicated unique_id vocab:   119,645
Vocab file written to /Users/ereboul/projects/smiles_blocks/data/processed/molgpt_data/block_vocab.txt


In [19]:
smiles_sel_df = successful_blocking[["smiles", "scaffold", "split"]].dropna(subset=[ "split"])

output_path = molgpt_dir / "molgpt_smiles_selection.csv"
smiles_sel_df.to_csv(output_path, index=False)
print(f"Written {len(smiles_sel_df):,} rows to {output_path}")
print(smiles_sel_df["split"].value_counts())

Written 1,928,855 rows to /Users/ereboul/projects/smiles_blocks/data/processed/molgpt_data/molgpt_smiles_selection.csv
split
train             1577803
test_scaffolds     175731
test               175321
Name: count, dtype: int64
